In [ ]:
##  %matplotlib inline
%matplotlib widget

import matplotlib as mpl
mpl.rcParams['animation.html'] = 'jshtml'
mpl.rcParams['animation.embed_limit'] = 200.0

import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import happi

In [ ]:
path = "../work_dir/tuto_3d"
path = "../work_dir/tuto_3d_PML"
# path = "../work_dir/tuto_3d_PML3b"
# path = "../work_dir/tuto_3d_PML"
field_component = "Bz"
axes_aspect = 'auto' # 'equal'


S = happi.Open(path, verbose=False)
print(S.namelist.Main.timestep) 
print(S.namelist.Main.geometry)
for species in S.namelist.Species:
    print("species "+species.name+" has mass "+str(species.mass))

Diag = S.Field(0,"Ex")
xxx = Diag.getData()
print(np.shape(xxx))

In [ ]:
%matplotlib widget
plt.close('all')
Lz = S.namelist.Main.grid_length[2]

Diag = S.Field(
    0,                      # diagnostic number
    "Ex",                   # field name

    # Take a slice at fixed z
    subset = {
        "z": Lz/2.
    },

    vmin = -0.1,
    vmax =  0.1,
)

Diag.slide()

plt.show()

In [ ]:
%matplotlib widget
plt.close('all')

Ex = S.Field(
    0,
    "Ex",
    subset={"z": Lz/2.},
    vmin=-0.05,
    vmax=0.05
)

Ey = S.Field(
    0,
    "Ey",
    subset={"z": Lz/2.},
    vmin=-0.05,
    vmax=0.05
)

Ez = S.Field(
    0,
    "Ez",
    subset={"z": Lz/2.},
    vmin=-0.05,
    vmax=0.05
)

happi.multiSlide(Ex, Ey, Ez,shape=[1,3])

plt.show()

In [ ]:
%matplotlib widget

import happi
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.widgets import Slider

S = happi.Open(path, verbose=False)

# -----------------------------------------
# Simulation geometry
# -----------------------------------------

Lz = S.namelist.Main.grid_length[2]
dz = S.namelist.Main.cell_length[2]

Nz = int(round(Lz/dz))

# Available z positions
z_values = np.arange(Nz)*dz

component = "Ex"

# Available timesteps
F0 = S.Field(0, component)
timesteps = F0.getTimesteps()

# -----------------------------------------
# Initial values
# -----------------------------------------

itime = 0
iz = Nz//2

# Initial diagnostic
Diag = S.Field(
    0,
    component,
    timesteps=timesteps[itime],
    subset={"z":[z_values[iz]]}
)

data = Diag.getData()[0]

# -----------------------------------------
# Figure
# -----------------------------------------

fig, ax = plt.subplots(figsize=(7,6))
plt.subplots_adjust(bottom=0.25)

# Initial color scale
vmax = np.max(np.abs(data))
vmin = -vmax

im = ax.imshow(
    data.T,
    origin='lower',
    aspect='auto',
    vmin=vmin,
    vmax=vmax
)

# Dynamic colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Ez")

title = ax.set_title(
    f"t = {timesteps[itime]}, z = {z_values[iz]:.2f}"
)

ax.set_xlabel("x")
ax.set_ylabel("y")

# -----------------------------------------
# Sliders
# -----------------------------------------

ax_time = plt.axes([0.15, 0.10, 0.7, 0.03])
ax_z    = plt.axes([0.15, 0.05, 0.7, 0.03])

slider_time = Slider(
    ax_time,
    'time',
    0,
    len(timesteps)-1,
    valinit=itime,
    valstep=1
)

slider_z = Slider(
    ax_z,
    'z-index',
    0,
    Nz-1,
    valinit=iz,
    valstep=1
)

# -----------------------------------------
# Update function
# -----------------------------------------

def update(val):

    it = int(slider_time.val)
    iz = int(slider_z.val)

    t = timesteps[it]
    z = z_values[iz]

    D = S.Field(
        0,
        component,
        timesteps=t,
        subset={"z":[z]}
    )

    new_data = D.getData()[0]

    im.set_data(new_data.T)

    vmax = np.max(np.abs(new_data))
    vmin = -vmax

    im.set_clim(vmin, vmax)

    # Update colorbar
    cbar.update_normal(im)

    title.set_text(
        f"t = {t}, z = {z:.2f}"
    )

    # im.autoscale()

    fig.canvas.draw_idle()

slider_time.on_changed(update)
slider_z.on_changed(update)

plt.show()

In [ ]:
# x-axis scan

%matplotlib widget

import happi
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.widgets import Slider

S = happi.Open(path, verbose=False)

# -----------------------------------------
# Simulation geometry
# -----------------------------------------

Lx = S.namelist.Main.grid_length[0]
dx = S.namelist.Main.cell_length[0]

Nx = int(round(Lx/dx))

# Available z positions
x_values = np.arange(Nx)*dx

component = "Ex"

# Available timesteps
F0 = S.Field(0, component)
timesteps = F0.getTimesteps()

# -----------------------------------------
# Initial values
# -----------------------------------------

itime = 0
ix = Nx//2

# Initial diagnostic
Diag = S.Field(
    0,
    component,
    timesteps=timesteps[itime],
    subset={"x":[x_values[ix]]}
)

data = Diag.getData()[0]

# -----------------------------------------
# Figure
# -----------------------------------------

fig, ax = plt.subplots(figsize=(7,6))
plt.subplots_adjust(bottom=0.25)

# Initial color scale
vmax = np.max(np.abs(data))
vmin = -vmax

im = ax.imshow(
    data.T,
    origin='lower',
    aspect='auto',
    vmin=vmin,
    vmax=vmax
)

# Dynamic colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Ez")

title = ax.set_title(
    f"t = {timesteps[itime]}, x = {x_values[ix]:.2f}"
)

# -----------------------------------------
# Sliders
# -----------------------------------------

ax_time = plt.axes([0.15, 0.10, 0.7, 0.03])
ax_x    = plt.axes([0.15, 0.05, 0.7, 0.03])

slider_time = Slider(
    ax_time,
    'time',
    0,
    len(timesteps)-1,
    valinit=itime,
    valstep=1
)

slider_x = Slider(
    ax_x,
    'x-index',
    0,
    Nx-1,
    valinit=ix,
    valstep=1
)

# -----------------------------------------
# Update function
# -----------------------------------------

def update(val):

    it = int(slider_time.val)
    ix = int(slider_x.val)

    t = timesteps[it]
    x = x_values[ix]

    D = S.Field(
        0,
        component,
        timesteps=t,
        subset={"x":[x]}
    )

    new_data = D.getData()[0]

    im.set_data(new_data.T)

    vmax = np.max(np.abs(new_data))
    vmin = -vmax

    im.set_clim(vmin, vmax)

    # Update colorbar
    cbar.update_normal(im)

    title.set_text(
        f"t = {t}, x = {x:.2f}"
    )

    # im.autoscale()

    fig.canvas.draw_idle()

slider_time.on_changed(update)
slider_x.on_changed(update)

plt.show()

In [ ]:
%matplotlib widget

import happi
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.widgets import Slider

# =========================================================
# USER OPTIONS
# =========================================================

FIELD = "Ex"

dynamic_scaling = False
# True  -> rescale every frame
# False -> fixed normalization

cmap = "RdBu_r"

# =========================================================
# OPEN SIMULATION
# =========================================================

S = happi.Open(path, verbose=False)

# =========================================================
# GEOMETRY
# =========================================================

Lz = S.namelist.Main.grid_length[2]
dz = S.namelist.Main.cell_length[2]

Nz = int(round(Lz/dz))

z_values = np.arange(Nz)*dz

# =========================================================
# TIMESTEPS
# =========================================================

F0 = S.Field(0, FIELD)
timesteps = F0.getTimesteps()

# Optional timestep reduction
# timesteps = timesteps[::10]

# =========================================================
# INITIAL POSITION
# =========================================================

itime = 0
iz = Nz//2

# =========================================================
# GLOBAL NORMALIZATION
# =========================================================

if not dynamic_scaling:

    print("Computing global normalization...")

    global_max = 0.

    for t in timesteps:

        D = S.Field(
            0,
            FIELD,
            timesteps=t,
            subset={"z":[z_values[iz]]}
        )

        data = D.getData()[0]

        local_max = np.max(np.abs(data))

        if local_max > global_max:
            global_max = local_max

    print("Global max =", global_max)

# =========================================================
# INITIAL DATA
# =========================================================

Diag = S.Field(
    0,
    FIELD,
    timesteps=timesteps[itime],
    subset={"z":[z_values[iz]]}
)

data = Diag.getData()[0]

# =========================================================
# INITIAL COLOR SCALE
# =========================================================

if dynamic_scaling:

    vmax = np.max(np.abs(data))

else:

    vmax = global_max

vmin = -vmax

# =========================================================
# FIGURE
# =========================================================

fig, ax = plt.subplots(figsize=(8,6))

plt.subplots_adjust(bottom=0.25)

im = ax.imshow(
    data.T,
    origin='lower',
    aspect='auto',
    cmap=cmap,
    vmin=vmin,
    vmax=vmax
)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label(FIELD)

title = ax.set_title(
    f"{FIELD} | t={timesteps[itime]} | z={z_values[iz]:.2f}"
)

ax.set_xlabel("x")
ax.set_ylabel("y")

# =========================================================
# SLIDERS
# =========================================================

ax_time = plt.axes([0.15, 0.10, 0.7, 0.03])
ax_z    = plt.axes([0.15, 0.05, 0.7, 0.03])

slider_time = Slider(
    ax_time,
    'time index',
    0,
    len(timesteps)-1,
    valinit=itime,
    valstep=1
)

slider_z = Slider(
    ax_z,
    'z index',
    0,
    Nz-1,
    valinit=iz,
    valstep=1
)

# =========================================================
# UPDATE FUNCTION
# =========================================================

def update(val):

    it = int(slider_time.val)
    iz = int(slider_z.val)

    t = timesteps[it]
    z = z_values[iz]

    D = S.Field(
        0,
        FIELD,
        timesteps=t,
        subset={"z":[z]}
    )

    new_data = D.getData()[0]

    # -----------------------------------------
    # Update image
    # -----------------------------------------

    im.set_data(new_data.T)

    # -----------------------------------------
    # Color scaling
    # -----------------------------------------

    if dynamic_scaling:

        vmax = np.max(np.abs(new_data))

    else:

        vmax = global_max

    vmin = -vmax

    im.set_clim(vmin, vmax)

    # Update colorbar
    cbar.update_normal(im)

    # -----------------------------------------
    # Update title
    # -----------------------------------------

    title.set_text(
        f"{FIELD} | t={t} | z={z:.2f}"
    )

    fig.canvas.draw_idle()

# =========================================================
# CONNECT
# =========================================================

slider_time.on_changed(update)
slider_z.on_changed(update)

plt.show()

In [ ]:
print([foo for foo in ['a','b','c'] if foo != 'b'][1])

In [ ]:
%matplotlib widget

import happi
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.widgets import Slider

# =========================================================
# USER OPTIONS
# =========================================================

FIELD = "Ex"
AXIS = "y"

dynamic_scaling = False
# True  -> rescale every frame
# False -> fixed normalization

cmap = "RdBu_r"

# =========================================================
# OPEN SIMULATION
# =========================================================

S = happi.Open(path, verbose=False)

# =========================================================
# GEOMETRY
# =========================================================

ax_idx = {"x":0,"y":1,"z":2}

L_spatial = S.namelist.Main.grid_length[ax_idx[AXIS]]
d_spatial = S.namelist.Main.cell_length[ax_idx[AXIS]]

N_spatial = int(round(L_spatial/d_spatial))

spatial_values = np.arange(N_spatial)*d_spatial

# =========================================================
# TIMESTEPS
# =========================================================

F0 = S.Field(0, FIELD)
timesteps = F0.getTimesteps()

# Optional timestep reduction
# timesteps = timesteps[::10]

# =========================================================
# INITIAL POSITION
# =========================================================

itime = 0
i_spatial = N_spatial//2

# =========================================================
# GLOBAL NORMALIZATION
# =========================================================

if not dynamic_scaling:

    print("Computing global normalization...")

    global_max = 0.

    for t in timesteps:

        D = S.Field(
            0,
            FIELD,
            timesteps=t,
            subset={AXIS:[spatial_values[i_spatial]]}
        )

        data = D.getData()[0]

        local_max = np.max(np.abs(data))

        if local_max > global_max:
            global_max = local_max

    print("Global max =", global_max)

# =========================================================
# INITIAL DATA
# =========================================================

Diag = S.Field(
    0,
    FIELD,
    timesteps=timesteps[itime],
    subset={AXIS:[spatial_values[i_spatial]]}
)

data = Diag.getData()[0]

# =========================================================
# INITIAL COLOR SCALE
# =========================================================

if dynamic_scaling:

    vmax = np.max(np.abs(data))

else:

    vmax = global_max

vmin = -vmax

# =========================================================
# FIGURE
# =========================================================

fig, ax = plt.subplots(figsize=(8,6))

plt.subplots_adjust(bottom=0.25)

im = ax.imshow(
    data.T,
    origin='lower',
    aspect='auto',
    cmap=cmap,
    vmin=vmin,
    vmax=vmax
)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label(FIELD)

title = ax.set_title(
    f"{FIELD} | t={timesteps[itime]} | "+AXIS+f"={spatial_values[i_spatial]:.2f}"
)


label_names = [foo for foo in ax_idx.keys() if foo != AXIS]
ax.set_xlabel(label_names[0])
ax.set_ylabel(label_names[1])

# =========================================================
# SLIDERS
# =========================================================

ax_time = plt.axes([0.15, 0.10, 0.7, 0.03])
ax_spatial    = plt.axes([0.15, 0.05, 0.7, 0.03])

slider_time = Slider(
    ax_time,
    'time index',
    0,
    len(timesteps)-1,
    valinit=itime,
    valstep=1
)

slider_spatial = Slider(
    ax_spatial,
    AXIS + ' index',
    0,
    N_spatial-1,
    valinit=i_spatial,
    valstep=1
)

# =========================================================
# UPDATE FUNCTION
# =========================================================

def update(val):

    it = int(slider_time.val)
    i_spatial = int(slider_spatial.val)

    t = timesteps[it]
    spatial_coordinate = spatial_values[i_spatial]

    D = S.Field(
        0,
        FIELD,
        timesteps=t,
        subset={AXIS:[spatial_coordinate]}
    )

    new_data = D.getData()[0]

    # -----------------------------------------
    # Update image
    # -----------------------------------------

    im.set_data(new_data.T)

    # -----------------------------------------
    # Color scaling
    # -----------------------------------------

    if dynamic_scaling:

        vmax = np.max(np.abs(new_data))

    else:

        vmax = global_max

    vmin = -vmax

    im.set_clim(vmin, vmax)

    # Update colorbar
    cbar.update_normal(im)

    # -----------------------------------------
    # Update title
    # -----------------------------------------

    title.set_text(
        f"{FIELD} | t={t} | "+AXIS+f"={spatial_coordinate:.2f}"
    )

    fig.canvas.draw_idle()

# =========================================================
# CONNECT
# =========================================================

slider_time.on_changed(update)
slider_spatial.on_changed(update)

plt.show()